# Part 2: Fine-tuning + Inference Optimization
**LLMForge | IISc LLM Course**

This notebook covers:
- LoRA implemented from scratch
- Instruction fine-tuning
- FP16, 4-bit quantization, KV cache benchmarks

In [1]:
import sys; sys.path.insert(0, '..')
import torch, time, math
import pandas as pd
import matplotlib.pyplot as plt
from src.data.tokenizer import CustomTokenizer
from src.model import TransformerLM, ModelConfig
from src.training.lora import LoRAConfig, inject_lora, save_adapter, load_adapter, merge_lora

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cpu


## 2.1 LoRA from Scratch

We implement LoRA manually — no `peft` library.
This shows exactly what happens inside `peft` under the hood.

In [2]:
# Load pre-trained model (from part 1 or fresh)
try:
    model = TransformerLM.load('data/checkpoints/part1_model.pt', device=device)
    print('Loaded Part 1 checkpoint')
except:
    model = TransformerLM(ModelConfig(
        vocab_size=50257, d_model=256, n_heads=8, n_layers=6, d_ff=1024
    ))
    print('Using fresh model')

# Show parameter count before LoRA
total_before = sum(p.numel() for p in model.parameters())
print(f'\nBefore LoRA: {total_before/1e6:.2f}M params')

[DEBUG] Building pos_encoding:learned with kwargs={'d_model': 128, 'd_head': 32, 'max_seq': 256, 'n_heads': 4, 'dropout': 0.1, 'theta': 10000.0}
[DEBUG] Building norm:layernorm with kwargs={'d_model': 128}
[DEBUG] Building norm:layernorm with kwargs={'d_model': 128}
[DEBUG] Building attention:standard with kwargs={'d_model': 128, 'n_heads': 4, 'n_kv_heads': None, 'dropout': 0.1}
[DEBUG] Building activation:gelu with kwargs={'d_model': 128, 'd_ff': 512}
[DEBUG] Building norm:layernorm with kwargs={'d_model': 128}
[DEBUG] Building norm:layernorm with kwargs={'d_model': 128}
[DEBUG] Building attention:standard with kwargs={'d_model': 128, 'n_heads': 4, 'n_kv_heads': None, 'dropout': 0.1}
[DEBUG] Building activation:gelu with kwargs={'d_model': 128, 'd_ff': 512}
[DEBUG] Building norm:layernorm with kwargs={'d_model': 128}
[DEBUG] Building norm:layernorm with kwargs={'d_model': 128}
[DEBUG] Building attention:standard with kwargs={'d_model': 128, 'n_heads': 4, 'n_kv_heads': None, 'dropout':

In [3]:
# Inject LoRA — experiment with different ranks
for rank in [4, 8, 16]:
    import copy
    m_test = copy.deepcopy(model)
    cfg    = LoRAConfig(
        rank=rank,
        alpha=rank * 2,
        dropout=0.05,
        target_modules=['qkv', 'out_proj']
    )
    m_test = inject_lora(m_test, cfg)
    trainable = sum(p.numel() for p in m_test.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in m_test.parameters())
    print(f'  Rank {rank:>2}: {trainable:>8,} trainable ({100*trainable/total:.3f}%)')

LoRA injected: 8 layers | Trainable: 7,003,392/7,271,296 (96.316%)
  Rank  4: 7,003,392 trainable (96.316%)
LoRA injected: 8 layers | Trainable: 7,015,680/7,283,584 (96.322%)
  Rank  8: 7,015,680 trainable (96.322%)
LoRA injected: 8 layers | Trainable: 7,040,256/7,308,160 (96.334%)
  Rank 16: 7,040,256 trainable (96.334%)


In [4]:
# Fine-tune with LoRA
from src.training.trainer import Trainer, TrainConfig
from src.data.loaders import make_dataloaders
from src.data.tokenizer import CustomTokenizer

tok = CustomTokenizer.from_pretrained('gpt2')

# Inject LoRA into model
lora_cfg = LoRAConfig(rank=8, alpha=16, target_modules=['qkv', 'out_proj'])
model    = inject_lora(model, lora_cfg)

# Fine-tune on Alpaca instruction data
train_dl, val_dl = make_dataloaders(
    'alpaca', tok._tok,
    batch_size=4, block_size=256, max_samples=1000
)

ft_cfg = TrainConfig(
    optimizer='adamw', lr=2e-4,
    lr_schedule='cosine', warmup_steps=20,
    max_steps=200, batch_size=4,
    gradient_accumulation=4,  # effective batch=16
    eval_every=50, save_every=100,
    precision='fp16' if device == 'cuda' else 'fp32',
    save_dir='data/checkpoints'
)

trainer = Trainer(model, train_dl, val_dl, ft_cfg, device=device)
ft_metrics = trainer.train()

# Save adapter (tiny file!)
save_adapter(model, 'data/checkpoints/lora_adapter', lora_cfg)
print('\nLoRA adapter saved (only ~2MB vs full model!)')

Loaded tokenizer: gpt2 | vocab=50,257
LoRA injected: 8 layers | Trainable: 7,015,680/7,283,584 (96.322%)
[DEBUG] Building dataset:alpaca with kwargs={'tokenizer': GPT2Tokenizer(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}), 'block_size': 256, 'train_split': 0.9, 'max_samples': 1000}
Loading Alpaca...
Dataset: alpaca | Train: 900 | Val: 100
Trainer ready | optimizer=adamw | schedule=cosine | precision=fp32


/home/user/Downloads/LLMForge/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



────────────────────────────────────────────────────────────
  Training: 200 steps | batch=4 | grad_accum=4
────────────────────────────────────────────────────────────



/home/user/Downloads/LLMForge/notebooks/../src/training/trainer.py:282: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=self.dtype, enabled=self.use_amp):


  step    50 | loss 5.8658 | ppl 352.76 | grad 0.861 | lr 1.87e-04 | 1871 tok/s


/home/user/Downloads/LLMForge/notebooks/../src/training/trainer.py:247: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=self.dtype, enabled=self.use_amp):


  ✓ Val loss: 5.5712 | Val PPL: 262.75
  Saved checkpoint → data/checkpoints/ckpt_best.pt
  step   100 | loss 4.9669 | ppl 143.58 | grad 0.741 | lr 1.17e-04 | 1926 tok/s
  ✓ Val loss: 4.8318 | Val PPL: 125.44
  Saved checkpoint → data/checkpoints/ckpt_best.pt
  Saved checkpoint → data/checkpoints/ckpt_step_100.pt
  step   150 | loss 4.8804 | ppl 131.69 | grad 0.697 | lr 3.57e-05 | 1937 tok/s
  ✓ Val loss: 4.6117 | Val PPL: 100.65
  Saved checkpoint → data/checkpoints/ckpt_best.pt
  step   200 | loss 4.4314 | ppl 84.05 | grad 0.656 | lr 0.00e+00 | 2038 tok/s
  ✓ Val loss: 4.5778 | Val PPL: 97.30
  Saved checkpoint → data/checkpoints/ckpt_best.pt
  Saved checkpoint → data/checkpoints/ckpt_step_200.pt
  Saved checkpoint → data/checkpoints/ckpt_final.pt

────────────────────────────────────────────────────────────
  Training complete in 6.8 min
  Best val loss: 4.5778
────────────────────────────────────────────────────────────

Adapter saved → data/checkpoints/lora_adapter (0.0 MB, 16 ten

## 2.2 Inference Optimization

Benchmark FP32 vs FP16 vs 4-bit quantization.
Measure: throughput, latency, VRAM.

In [5]:
# Load Phi-2 or GPT-2 for optimization benchmark
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'gpt2'

results = []

# FP32 baseline
print('Testing FP32...')
model_fp32 = AutoModelForCausalLM.from_pretrained(MODEL_ID)
model_fp32 = model_fp32.to(device).eval()
base_tok   = AutoTokenizer.from_pretrained(MODEL_ID)
base_tok.pad_token = base_tok.eos_token

ids = base_tok.encode('The transformer architecture', return_tensors='pt').to(device)

if device == 'cuda': torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
t0 = time.time()
with torch.no_grad():
    model_fp32.generate(ids, max_new_tokens=50, do_sample=False, pad_token_id=base_tok.eos_token_id)
if device == 'cuda': torch.cuda.synchronize()
elapsed   = time.time() - t0
vram_fp32 = torch.cuda.max_memory_allocated()/1e6 if device=='cuda' else 0
ppl_fp32  = math.exp(model_fp32(ids, labels=ids).loss.item())
results.append({'Precision': 'FP32', 'tok/s': round(50/elapsed,1), 'VRAM MB': round(vram_fp32,0), 'PPL': round(ppl_fp32,2)})
del model_fp32; torch.cuda.empty_cache() if device=='cuda' else None
print(f'  FP32: {50/elapsed:.1f} tok/s | {vram_fp32:.0f} MB VRAM | PPL={ppl_fp32:.2f}')

Testing FP32...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


  FP32: 18.8 tok/s | 0 MB VRAM | PPL=73547.42


In [6]:
# FP16
print('Testing FP16...')
model_fp16 = AutoModelForCausalLM.from_pretrained(MODEL_ID).half().to(device).eval()

if device=='cuda': torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
t0 = time.time()
with torch.no_grad():
    model_fp16.generate(ids, max_new_tokens=50, do_sample=False, pad_token_id=base_tok.eos_token_id)
if device=='cuda': torch.cuda.synchronize()
elapsed   = time.time() - t0
vram_fp16 = torch.cuda.max_memory_allocated()/1e6 if device=='cuda' else 0
ppl_fp16  = math.exp(model_fp16(ids.to(device), labels=ids.to(device)).loss.item())
ppl_diff  = abs(ppl_fp16 - ppl_fp32)/ppl_fp32 * 100
results.append({'Precision': 'FP16', 'tok/s': round(50/elapsed,1), 'VRAM MB': round(vram_fp16,0), 'PPL': round(ppl_fp16,2)})
print(f'  FP16: {50/elapsed:.1f} tok/s | {vram_fp16:.0f} MB VRAM | PPL={ppl_fp16:.2f}')
print(f'  PPL difference: {ppl_diff:.3f}% ({"✓ PASS" if ppl_diff < 1.0 else "✗ FAIL"} < 1% threshold)')
del model_fp16; torch.cuda.empty_cache() if device=='cuda' else None

Testing FP16...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  FP16: 6.3 tok/s | 0 MB VRAM | PPL=72145.86
  PPL difference: 1.906% (✗ FAIL < 1% threshold)


In [7]:
# 4-bit quantization (QLoRA-style)
print('Testing 4-bit quantization...')
try:
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
    )
    model_4bit = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_cfg)
    if device=='cuda': torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        model_4bit.generate(ids, max_new_tokens=50, do_sample=False, pad_token_id=base_tok.eos_token_id)
    if device=='cuda': torch.cuda.synchronize()
    elapsed     = time.time() - t0
    vram_4bit   = torch.cuda.max_memory_allocated()/1e6 if device=='cuda' else 0
    results.append({'Precision': '4-bit', 'tok/s': round(50/elapsed,1), 'VRAM MB': round(vram_4bit,0), 'PPL': 'N/A'})
    print(f'  4-bit: {50/elapsed:.1f} tok/s | {vram_4bit:.0f} MB VRAM')
    del model_4bit; torch.cuda.empty_cache() if device=='cuda' else None
except Exception as e:
    print(f'  4-bit skipped: {e}')

# Summary table
df = pd.DataFrame(results)
print('\n' + '='*50)
print('INFERENCE OPTIMIZATION SUMMARY')
print('='*50)
print(df.to_string(index=False))

Testing 4-bit quantization...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/home/user/Downloads/LLMForge/venv/lib/python3.12/site-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/user/Downloads/LLMForge/venv/lib/python3.12/site-packages/bitsandbytes/backends/cpu/ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/user/Downloads/LLMForge/venv/lib/python3.12/site-packages/bitsandbytes/backends/cpu/ops.py:80: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/user/Downloads/LLMForge/venv/lib/python3.12/site-packages/bitsandbytes/backends/cpu/ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch relea

  4-bit: 3.4 tok/s | 0 MB VRAM

INFERENCE OPTIMIZATION SUMMARY
Precision  tok/s  VRAM MB       PPL
     FP32   18.8        0  73547.42
     FP16    6.3        0  72145.86
    4-bit    3.4        0       N/A


In [8]:
# Batch size scaling (throughput vs batch)
print('\nBatch Size Scaling (GPT-2, FP16)...')
model_batch = AutoModelForCausalLM.from_pretrained(MODEL_ID).half().to(device).eval()
batch_results = []

for bs in [1, 2, 4, 8]:
    try:
        b_ids = ids.repeat(bs, 1)
        if device=='cuda': torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
        t0 = time.time()
        with torch.no_grad():
            model_batch.generate(b_ids, max_new_tokens=50, do_sample=False, pad_token_id=base_tok.eos_token_id)
        if device=='cuda': torch.cuda.synchronize()
        elapsed = time.time() - t0
        vram    = torch.cuda.max_memory_allocated()/1e6 if device=='cuda' else 0
        tps     = bs*50/elapsed
        batch_results.append({'batch': bs, 'total_tok/s': round(tps,1),
                               'per_seq_tok/s': round(tps/bs,1), 'VRAM MB': round(vram,0)})
        print(f'  bs={bs}: {tps:.0f} total tok/s | {tps/bs:.0f} per-seq tok/s | {vram:.0f} MB')
    except RuntimeError as e:
        print(f'  bs={bs}: OOM - {str(e)[:60]}')
        break

del model_batch; torch.cuda.empty_cache() if device=='cuda' else None
pd.DataFrame(batch_results)


Batch Size Scaling (GPT-2, FP16)...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  bs=1: 6 total tok/s | 6 per-seq tok/s | 0 MB
  bs=2: 7 total tok/s | 3 per-seq tok/s | 0 MB
  bs=4: 7 total tok/s | 2 per-seq tok/s | 0 MB
  bs=8: 7 total tok/s | 1 per-seq tok/s | 0 MB


,batch,total_tok/s,per_seq_tok/s,VRAM MB
0,1,6.4,6.4,0
1,2,6.8,3.4,0
2,4,7.0,1.8,0
3,8,7.0,0.9,0
